<a href="https://colab.research.google.com/github/sitahlango-maker/Financial_Inclusion/blob/main/CleanFinancialinclusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Installing Basic Liberaries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Imports the pandas library for data handling and analysis.
import pandas as pd

In [2]:
# Loads the Kenya survey data file from the public GitHub repository.
df_ken = pd.read_csv('https://raw.githubusercontent.com/sitahlango-maker/Financial_Inclusion/main/Colab%20Notebooks/FinancialInclution/Findex_Microdata_2025_Kenya.csv')

# Loads the Tanzania survey data file from the public GitHub repository.
df_tza = pd.read_csv('https://raw.githubusercontent.com/sitahlango-maker/Financial_Inclusion/main/Colab%20Notebooks/FinancialInclution/Findex_Microdata_2025_Tanzania.csv')

# Loads the Uganda survey data file from the public GitHub repository.
df_uga = pd.read_csv('https://raw.githubusercontent.com/sitahlango-maker/Financial_Inclusion/main/Colab%20Notebooks/FinancialInclution/Findex_Microdata_2025_Uganda.csv')

In [3]:
# Prints the number of rows and columns for each country’s survey file to confirm successful loading.
print("Kenya shape:", df_ken.shape)
print("Tanzania shape:", df_tza.shape)
print("Uganda shape:", df_uga.shape)

# Shows the first 15 column names of the Kenya file to check the structure.
print("\nKenya columns:", df_ken.columns.tolist()[:15], "...")

Kenya shape: (1000, 183)
Tanzania shape: (1000, 183)
Uganda shape: (1000, 183)

Kenya columns: ['year', 'economy', 'economycode', 'regionwb', 'pop_adult', 'wpid_random', 'wgt', 'female', 'age', 'educ', 'inc_q', 'emp_in', 'urbanicity', 'account_fin', 'account_mob'] ...


**Cleaning** **initial** **Findex** **Dataset**

In [4]:
# Defines a function to clean one Findex survey file.
def clean_findex(df, country_code, country_name):

    # Creates a copy of the input DataFrame to avoid changing the original.
    df = df.copy()

    # Adds the full country name as a new column.
    df['country'] = country_name

    # Adds the three-letter country code as a new column.
    df['country_code'] = country_code

    # Checks if the survey weight column exists and converts it to numeric values.
    if 'wgt' in df.columns:
        df['wgt'] = pd.to_numeric(df['wgt'], errors='coerce')

    # Defines the main outcome columns that measure mobile money and digital use.
    target_cols = ['account_mob', 'dig_account', 'anydigpayment']

    # Loops through each outcome column and converts it to numeric values.
    for col in target_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Lists the most important columns to keep in the cleaned file.
    keep = ['country', 'country_code', 'wgt', 'female', 'age', 'educ', 'inc_q',
            'urbanicity', 'account_mob', 'dig_account', 'anydigpayment', 'internet_use']

    # Removes any listed columns that do not exist in the DataFrame.
    keep = [c for c in keep if c in df.columns]

    # Keeps only the selected columns and discards the rest.
    df = df[keep]

    # Returns the cleaned DataFrame.
    return df

# Loads the Kenya survey file directly from the public GitHub repository.
df_ken = pd.read_csv('https://raw.githubusercontent.com/sitahlango-maker/Financial_Inclusion/main/Colab%20Notebooks/FinancialInclution/Findex_Microdata_2025_Kenya.csv')

# Loads the Tanzania survey file directly from the public GitHub repository.
df_tza = pd.read_csv('https://raw.githubusercontent.com/sitahlango-maker/Financial_Inclusion/main/Colab%20Notebooks/FinancialInclution/Findex_Microdata_2025_Tanzania.csv')

# Loads the Uganda survey file directly from the public GitHub repository.
df_uga = pd.read_csv('https://raw.githubusercontent.com/sitahlango-maker/Financial_Inclusion/main/Colab%20Notebooks/FinancialInclution/Findex_Microdata_2025_Uganda.csv')

# Cleans the Kenya data using the function defined earlier.
df_ken_clean = clean_findex(df_ken, 'KEN', 'Kenya')

# Cleans the Tanzania data using the function defined earlier.
df_tza_clean = clean_findex(df_tza, 'TZA', 'Tanzania')

# Cleans the Uganda data using the function defined earlier.
df_uga_clean = clean_findex(df_uga, 'UGA', 'Uganda')

# Combines the three cleaned files into one table (df_micro).
df_micro = pd.concat([df_ken_clean, df_tza_clean, df_uga_clean], ignore_index=True)

# Prints a confirmation message with the shape of the combined table.
print("df_micro defined. Shape:", df_micro.shape)

df_micro defined. Shape: (3000, 12)


In [5]:
# Cleans Kenya survey data using the cleaning function.
df_ken_clean = clean_findex(df_ken, 'KEN', 'Kenya')

# Cleans Tanzania survey data using the cleaning function.
df_tza_clean = clean_findex(df_tza, 'TZA', 'Tanzania')

# Cleans Uganda survey data using the cleaning function.
df_uga_clean = clean_findex(df_uga, 'UGA', 'Uganda')

# Combines three cleaned files into one single table.
df_micro = pd.concat([df_ken_clean, df_tza_clean, df_uga_clean], ignore_index=True)

# Prints total rows and columns of combined table.
print("Combined microdata shape:", df_micro.shape)

# Shows how many rows belong to each country.
print(df_micro['country'].value_counts())

# Prints percentage of missing values in each column (top 12).
print("\nMissing values (%):\n", df_micro.isna().mean().sort_values(ascending=False).head(12))

Combined microdata shape: (3000, 12)
country
Kenya       1000
Tanzania    1000
Uganda      1000
Name: count, dtype: int64

Missing values (%):
 educ             0.001
country          0.000
wgt              0.000
country_code     0.000
female           0.000
age              0.000
inc_q            0.000
urbanicity       0.000
account_mob      0.000
dig_account      0.000
anydigpayment    0.000
internet_use     0.000
dtype: float64


**Loading the country level dataset**

In [6]:
# Loads prevalence index file .
df_preval = pd.read_csv('https://raw.githubusercontent.com/sitahlango-maker/Financial_Inclusion/refs/heads/main/Colab%20Notebooks/FinancialInclution/Mobile%20Money%20Prevalent%20Index-2020-23-Public(MMPI%202020-23).csv')

# Keeps only needed columns and removes rows without valid code.
df_preval = df_preval[['Country', 'ISO3', 'Mobile Money Prevalence (2023)']].dropna(subset=['ISO3'])

# Changes column names to be short and clear.
df_preval.columns = ['country_name', 'country_code', 'mmpi_2023']

# Keeps only Kenya, Tanzania, and Uganda rows.
df_preval = df_preval[df_preval['country_code'].isin(['KEN', 'TZA', 'UGA'])]

In [7]:
# Loads regulatory index file
df_reg = pd.read_csv('https://raw.githubusercontent.com/sitahlango-maker/Financial_Inclusion/main/Colab%20Notebooks/FinancialInclution/Mobile_Money_Regulatory_Index_Database_2025_v2(Data).csv')

# Keeps only rows from year 2025 (latest data).
df_reg = df_reg[df_reg['Year'] == 2025]

# Keeps only selected columns that are most relevant.
df_reg = df_reg[['Country', 'Index', 'Consumer Protection', 'KYC Proportionality',
                 'Entry-level transaction limits', 'Maximum transaction limits', 'Agent Eligibility']]

# Changes column names to be short and clear.
df_reg.columns = ['country_name', 'reg_index', 'reg_cons_prot', 'reg_kyc_prop',
                  'reg_entry_lim', 'reg_max_lim', 'reg_agent_el']

# Adds short country code using country name.
df_reg['country_code'] = df_reg['country_name'].map({'Kenya': 'KEN', 'Tanzania': 'TZA', 'Uganda': 'UGA'})

# Keeps only Kenya, Tanzania, and Uganda rows.
df_reg = df_reg[df_reg['country_code'].isin(['KEN', 'TZA', 'UGA'])]

In [8]:
# Loads deployment tracker file
df_deploy = pd.read_csv('https://raw.githubusercontent.com/sitahlango-maker/Financial_Inclusion/refs/heads/main/Colab%20Notebooks/FinancialInclution/Mobile%20Money%20Deployment.csv')

# Keeps only rows for Kenya, Tanzania, and Uganda.
df_deploy = df_deploy[df_deploy['Country ISO Code'].isin(['KEN', 'TZA', 'UGA'])]

# Counts number of mobile money providers per country.
df_providers = df_deploy.groupby('Country ISO Code').size().reset_index(name='num_providers')

# Changes country code column name to match others.
df_providers = df_providers.rename(columns={'Country ISO Code': 'country_code'})

# Changes launch year to numbers (ignores errors).
df_deploy['launch_year'] = pd.to_numeric(df_deploy['Launch Year'], errors='coerce')

# Finds earliest launch year per country.
df_oldest = df_deploy.groupby('Country ISO Code')['launch_year'].min().reset_index(name='earliest_launch_year')

# Changes country code column name to match others.
df_oldest = df_oldest.rename(columns={'Country ISO Code': 'country_code'})

**Building Country Facts Table**

In [9]:
# Starts country facts table with prevalence category.
df_country_facts = df_preval[['country_code', 'mmpi_2023']].copy()

# Adds regulatory scores using left join.
df_country_facts = df_country_facts.merge(
    df_reg[['country_code', 'reg_index', 'reg_cons_prot', 'reg_kyc_prop',
            'reg_entry_lim', 'reg_max_lim', 'reg_agent_el']],
    on='country_code',
    how='left'
)

# Combines provider count and earliest launch year.
df_deploy_info = df_providers.merge(df_oldest, on='country_code', how='left')

# Adds provider information to country facts table.
df_country_facts = df_country_facts.merge(df_deploy_info, on='country_code', how='left')

# Prints the final country facts table.
print(df_country_facts)

  country_code  mmpi_2023  reg_index  reg_cons_prot  reg_kyc_prop  \
0          KEN  Very high      88.00         100.00             0   
1          TZA  Very high      87.16          83.33           100   
2          UGA  Very high      88.33         100.00           100   

   reg_entry_lim  reg_max_lim  reg_agent_el  num_providers  \
0            100          100           100              4   
1            100          100           100              6   
2            100          100           100              7   

   earliest_launch_year  
0                  2007  
1                  2008  
2                  2009  


**Selecting Survey Columns and Final Combination**

In [10]:
# Lists most useful columns from survey data.
keep_survey = [
    'country_code', 'female', 'age', 'educ', 'inc_q', 'urbanicity',
    'account_mob', 'dig_account', 'anydigpayment', 'internet_use', 'wgt'
]

# Keeps only selected columns from combined survey data.
df_survey_clean = df_micro[keep_survey].copy()

# Joins survey data with country facts using country code.
df_final = df_survey_clean.merge(df_country_facts, on='country_code', how='left')

# Prints final table size to confirm.
print("Final combined dataset shape:", df_final.shape)

# Prints first few rows of final table.
print("First few rows:\n", df_final.head())

# Prints percentage of missing values in each column.
print("\nMissing values (%):\n", df_final.isna().mean().sort_values(ascending=False).head(10))

Final combined dataset shape: (3000, 20)
First few rows:
   country_code  female  age  educ  inc_q  urbanicity  account_mob  \
0          KEN       1   25   2.0      1           1            1   
1          KEN       1   26   2.0      4           1            1   
2          KEN       1   21   2.0      3           1            1   
3          KEN       1   25   2.0      5           1            1   
4          KEN       2   31   3.0      5           2            1   

   dig_account  anydigpayment  internet_use       wgt  mmpi_2023  reg_index  \
0            1              1             1  0.723252  Very high       88.0   
1            1              1             1  0.331405  Very high       88.0   
2            1              1             1  1.071302  Very high       88.0   
3            1              1             1  0.677005  Very high       88.0   
4            1              1             1  0.457662  Very high       88.0   

   reg_cons_prot  reg_kyc_prop  reg_entry_lim  reg_m

**Saving the Final Dataset as CSV**

In [ ]:
# Save the dataset as CSV in Colab's temporary storage
df_final.to_csv('FinalCombine.csv', index=False)

# Install git if not already available (usually pre-installed in Colab)
!apt-get update -qq && apt-get install -y git

# Configure git with your details (replace with your own email and username)
!git config --global user.email "your-email@example.com"
!git config --global user.name "Your GitHub Username"

# Clone your repository (replace with your actual repo URL)
!git clone https://github.com/sitahlango-maker/Financial_Inclusion.git
%cd Financial_Inclusion

# Move the CSV file to the desired folder inside the repo
!mkdir -p "Colab Notebooks/FinancialInclution"
!mv ../FinalCombine.csv "Colab Notebooks/FinancialInclution/FinalCombine.csv"

# Add, commit, and push the file to GitHub
!git add "Colab Notebooks/FinancialInclution/FinalCombine.csv"
!git commit -m "Add FinalCombine.csv - combined Findex and country-level dataset"
!git push origin main

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Failed to fetch https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu/dists/jammy/InRelease  Could not connect to ppa.launchpadcontent.net:443 (185.125.190.80). - connect (111: Connection refused)
W: Failed to fetch https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu/dists/jammy/InRelease  Unable to connect to ppa.launchpadcontent.net:443:
W: Some index files failed to download. They have been ignored, or old ones used instead.


In [ ]:
# Load the Mobile Money Deployment Tracker
df_preval = pd.read_csv('https://raw.githubusercontent.com/sitahlango-maker/Financial_Inclusion/refs/heads/main/Colab%20Notebooks/FinancialInclution/Mobile%20Money%20Prevalent%20Index-2020-23-Public(MMPI%202020-23).csv',
 )

# Rename column for merging
df_preval = df_preval[['Country', 'ISO3', 'Mobile Money Prevalence (2023)']]

#--Remove rows without valid country code
df_preval = df_preval.dropna(subset=['ISO3'])

#Select the columns to be used:
df_preval = df_preval[['Country', 'ISO3', 'Mobile Money Prevalence (2023)']]

#Make the column names short and clear
df_preval.columns = ['country_name', 'country_code', 'mmpi_2023']


#Keep only three countries (Kenya, Uganda and Tanzania)
df_preval = df_preval[df_preval['country_code'].isin(['KEN', 'TZA', 'UGA'])]

df_preval.columns.tolist()

**Combining the four latter GSMA datasets With the Original Findex Dataset**

In [ ]:
df_country_facts = df_preval[['country_code', 'mmpi_2023']].copy()
df_country_facts = df_country_facts.merge(df_reg[['country_code', 'reg_index', 'reg_cons_prot', 'reg_kyc_prop',
                                                  'reg_entry_lim', 'reg_max_lim', 'reg_agent_el']],
                                          on='country_code', how='left')
df_deploy_info = df_providers.merge(df_oldest, on='country_code', how='left')
df_country_facts = df_country_facts.merge(df_deploy_info, on='country_code', how='left')

In [ ]:
# Preparing microdata (the three Findex survey files already combined in df_micro)
# Adding country_code (just in case it's not updated properly)
df_micro['country_code'] = df_micro['country'].map({
    'Kenya': 'KEN',
    'Tanzania': 'TZA',
    'Uganda': 'UGA'
})

In [ ]:
# Keep only the most useful survey columns
keep_survey = [
    'country_code', 'female', 'age', 'educ', 'inc_q', 'urbanicity',
    'account_mob', 'dig_account', 'anydigpayment', 'internet_use', 'wgt'
]
df_survey_clean = df_micro[keep_survey].copy()

In [ ]:
# Building one small country facts table from the other five sources
# Starting with prevalence dataset as base
df_country_facts = df_preval[['country_code', 'mmpi_2023']].copy()

In [ ]:
# Adding regulatory scores
df_country_facts = df_country_facts.merge(
    df_reg[['country_code', 'reg_index', 'reg_cons_prot', 'reg_kyc_prop',
            'reg_entry_lim', 'reg_max_lim', 'reg_agent_el']],
    on='country_code',
    how='left'
)

In [ ]:
# Adding the number of providers and earliest launch year
df_providers = df_deploy.groupby('Country ISO Code').size().reset_index(name='num_providers')
df_earliest = df_deploy.groupby('Country ISO Code')['launch_year'].min().reset_index(name='earliest_launch')
df_providers = df_providers.rename(columns={'Country ISO Code': 'country_code'})
df_earliest  = df_earliest.rename(columns={'Country ISO Code': 'country_code'})

df_deploy_info = df_providers.merge(df_earliest, on='country_code', how='left')
df_country_facts = df_country_facts.merge(df_deploy_info, on='country_code', how='left')

In [ ]:
# Joining the country facts to every row of the survey data
df_final = df_survey_clean.merge(
    df_country_facts,
    on='country_code',
    how='left'
)

In [ ]:
# Checking the result
print("Final combined dataset shape:", df_final.shape)
print("First few rows:\n", df_final.head())
print("\nMissing values (%):\n", df_final.isna().mean().sort_values(ascending=False).head(10))

In [ ]:
# Saving the final file
df_final.to_csv(
    'https://github.com/sitahlango-maker/Financial_Inclusion/tree/main/Colab%20Notebooks/FinancialInclution',
    index=False
)

In [ ]:
# Import libraries needed for data preparation and modelling
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

In [ ]:
# Set display options for better readability
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

**Saving the Cleaned Raw Dataset**

In [ ]:
df_final.to_csv('cleaned_multiple_country_dataset.csv',index=False)

In [ ]:
# Display basic information about the loaded data
print("Dataset shape:", df_final.shape)
print("\nFirst few rows:")
print(df_final.head())
print("\nColumn names:")
print(df_final.columns.tolist())
print("\nMissing values (%):")
print(df_final.isna().mean().sort_values(ascending=False).head(10))


In [ ]:
# Imports pandas for loading and managing data tables.
import pandas as pd

# Imports numpy for numerical calculations and array handling.
import numpy as np

# Imports matplotlib and seaborn for creating plots and visual checks.
import matplotlib.pyplot as plt
import seaborn as sns

# Imports tools from scikit-learn for splitting data and building models.
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report


In [ ]:

# Prints the shape (rows and columns) of the loaded data to confirm it is correct.
print("Dataset shape:", df_final.shape)


In [ ]:

# Prints all column names to see what features are available.
print("\nColumn names:")
print(df_final.columns.tolist())

In [ ]:
# Prints the percentage of missing values in each column (sorted highest to lowest).
print("\nMissing values (%):")
print(df_final.isna().mean().sort_values(ascending=False).head(15))

In [ ]:
# Prints the data types of each column to check they are correct.
print("\nData types:")
print(df_final.dtypes)

In [ ]:
country_series = df_final['country_code'].copy()

**Splitting Global the dataset**

In [ ]:
# Global Train-Test Split + Defining X_train / X_test

from sklearn.model_selection import train_test_split

print("Performing global train-test split...\n")

# Define target and features
y = df_final['account_mob'].copy()                    # Target variable
X = df_final.drop(columns=['account_mob'])            # Drop only the target

# Keep country_code for later subsetting (very important!)
country_series = df_final['country_code'].copy()

# Train-test split (stratified by target)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

# Preserve country labels aligned with train/test sets
country_train = country_series.loc[X_train.index]
country_test  = country_series.loc[X_test.index]

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape : {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape : {y_test.shape}")
print(f"Country distribution in train:\n{country_train.value_counts()}")

In [ ]:
# Training Country-Specific Expert Models

from sklearn.ensemble import RandomForestClassifier
import joblib

experts = {}
expert_metrics = {}

countries = ['KEN', 'TZA', 'UGA']

print("Training country-specific expert models...\n")

for country in countries:
    print(f"→ Training expert for {country}")

    # Use mask based on country_train (safest way)
    mask_train = (country_train == country)

    if mask_train.sum() < 20:
        print(f"  Skipping {country} — too few training samples ({mask_train.sum()})")
        continue

    # Correct subsetting after encoding
    X_train_country = X_train[mask_train]
    y_train_country = y_train[mask_train]

    # Safety: ensure only numeric columns are used
    X_train_country = X_train_country.select_dtypes(include=['number', 'bool'])

    if X_train_country.empty:
        print(f"  Skipping {country} — no numeric features left")
        continue

    # Train the expert model
    model_expert = RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced'
    )

    model_expert.fit(X_train_country, y_train_country)

    experts[country] = model_expert

    print(f"  Trained successfully ({mask_train.sum()} samples)\n")

# Summary
print(f"Trained experts for: {list(experts.keys())}")
print(f"Countries skipped (if any): {[c for c in countries if c not in experts]}")

In [ ]:
# One-Hot Encoding (Leakage-safe)
# Identify categorical columns (exclude country_code and target)
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# Remove 'country_code' from encoding (we already have it in country_train/test)
if 'country_code' in categorical_cols:
    categorical_cols.remove('country_code')

print(f"Encoding categorical columns: {categorical_cols}")

# One-hot encode
X_train_encoded = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_test_encoded  = pd.get_dummies(X_test,  columns=categorical_cols, drop_first=True)

# Align columns (important!)
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

# Final assignment
X_train = X_train_encoded
X_test  = X_test_encoded

print(f"Final X_train shape after encoding: {X_train.shape}")
print(f"Final X_test shape after encoding : {X_test.shape}")
print("All columns are now numeric:", X_train.dtypes.unique())

**Training and Evaluating the pooled combined model**

In [ ]:
# Train & Evaluate Pooled (Combined) Model
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

print("Training Pooled (Combined) Model...\n")

In [ ]:
# IMPORTANT FIX: Remove country_code before training
X_train = X_train.drop(columns=['country_code'], errors='ignore')
X_test  = X_test.drop(columns=['country_code'], errors='ignore')

# Train the Pooled Model
model_pooled = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)

model_pooled.fit(X_train, y_train)

print("Pooled model trained successfully.")

In [ ]:
# Evaluate on Test Set
print("\nEvaluating Pooled Model on Test Set...\n")

pooled_preds = model_pooled.predict(X_test)
pooled_probs = model_pooled.predict_proba(X_test)[:, 1]

pooled_accuracy = accuracy_score(y_test, pooled_preds)
pooled_f1 = f1_score(y_test, pooled_preds, zero_division=0)
pooled_auc = roc_auc_score(y_test, pooled_probs)

print(f"Accuracy : {pooled_accuracy:.4f}")
print(f"F1-score : {pooled_f1:.4f}")
print(f"AUC      : {pooled_auc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, pooled_preds, zero_division=0))

**Evaluate all Models Fairly**

In [ ]:
# Evaluate All Models Fairly
# Pooled model vs Country-specific Expert model

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import pandas as pd

print("Evaluating All Models Fairly...\n")

results = []


In [ ]:

#  Pooled Model Evaluation (Full Test Set)
print("Evaluating Pooled Model...")

pooled_preds = model_pooled.predict(X_test)
pooled_probs = model_pooled.predict_proba(X_test)[:, 1]

results.append({
    'Model': 'Pooled',
    'Country': 'All',
    'Samples': len(y_test),
    'Accuracy': accuracy_score(y_test, pooled_preds),
    'F1': f1_score(y_test, pooled_preds, zero_division=0),
    'AUC': roc_auc_score(y_test, pooled_probs)
})

In [ ]:
# Pooled performance broken down by country
for country in ['KEN', 'TZA', 'UGA']:
    mask = (country_test == country)
    if mask.sum() == 0:
        continue
    results.append({
        'Model': 'Pooled',
        'Country': country,
        'Samples': mask.sum(),
        'Accuracy': accuracy_score(y_test[mask], pooled_preds[mask]),
        'F1': f1_score(y_test[mask], pooled_preds[mask], zero_division=0),
        'AUC': roc_auc_score(y_test[mask], pooled_probs[mask])
    })

In [ ]:

# Expert Models Evaluation (Only on their own country)
print("Evaluating Expert Models...")

for country, model in experts.items():
    mask = (country_test == country)
    if mask.sum() == 0:
        print(f"No test samples for {country}")
        continue

    X_test_c = X_test[mask]
    y_test_c = y_test[mask]

    preds = model.predict(X_test_c)
    probs = model.predict_proba(X_test_c)[:, 1]

    results.append({
        'Model': f'Expert_{country}',
        'Country': country,
        'Samples': mask.sum(),
        'Accuracy': accuracy_score(y_test_c, preds),
        'F1': f1_score(y_test_c, preds, zero_division=0),
        'AUC': roc_auc_score(y_test_c, probs)
    })

In [ ]:
# Create and Display Results Table
df_results = pd.DataFrame(results).round(4)

print("\n=== Model Performance Comparison ===")
print(df_results.to_string(index=False))

# Nice styled display
display(
    df_results.style
    .format(precision=4)
    .highlight_max(color='lightgreen', subset=['Accuracy', 'F1', 'AUC'])
    .set_caption("Fair Evaluation: Pooled vs Country Expert Models")
)

# Save to CSV
df_results.to_csv('model_comparison_results.csv', index=False)
print("\nResults saved to 'model_comparison_results.csv'")

**FEATURE IMPORTANCE COMPARISSON**

In [ ]:
# Feature Importance Comparison - Separate Subplots
# One subplot per model (Pooled + KEN + TZA + UGA)
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Generating Feature Importance - Separate Subplots...\n")

In [ ]:
# Collect feature importances
importance_dict = {}

# Pooled Model
if 'model_pooled' in globals():
    importance_dict['Pooled'] = pd.Series(
        model_pooled.feature_importances_,
        index=X_train.columns
    ).sort_values(ascending=False)

# Expert Models
for country, model in experts.items():
    if hasattr(model, 'feature_importances_'):
        importance_dict[f'Expert_{country}'] = pd.Series(
            model.feature_importances_,
            index=X_train.columns
        ).sort_values(ascending=False)


In [ ]:
# Plot: Separate Subplots
if not importance_dict:
    print("No models found.")
else:
    n_models = len(importance_dict)
    top_n = 10  # Show top 10 features per model

    # Create subplots (1 row, multiple columns)
    fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 8), sharey=False)

    if n_models == 1:
        axes = [axes]  # Make it iterable if only one model

    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']  # Blue, Orange, Green, Red

    for i, (model_name, imp_series) in enumerate(importance_dict.items()):
        # Get top N features
        top_features = imp_series.head(top_n)

        # Plot on respective subplot
        sns.barplot(
            ax=axes[i],
            x=top_features.values,
            y=top_features.index,
            color=colors[i % len(colors)]
        )

        axes[i].set_title(f'{model_name}', fontsize=14, pad=15)
        axes[i].set_xlabel('Importance', fontsize=12)
        if i == 0:
            axes[i].set_ylabel('Feature', fontsize=12)
        else:
            axes[i].set_ylabel('')

        # Add value labels on bars
        for j, v in enumerate(top_features.values):
            axes[i].text(v + 0.005, j, f'{v:.3f}', va='center', fontsize=10)

    plt.suptitle('Feature Importance Comparison: Pooled vs Country Expert Models\n(Top 10 Features)',
                 fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

    # Print Top 8 Features per Model (Text Summary)
    print("\nTop 8 Features per Model:")
    print("=" * 70)
    for model_name, series in importance_dict.items():
        print(f"\n{model_name}:")
        print(series.head(8).round(4).to_string())
        print("-" * 50)

**Accuracy by gender/rural/etc. for pooled vs experts**

In [ ]:
# Subgroup Accuracy Comparison: Pooled vs Expert Models
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

print("Computing Accuracy by Subgroups - Pooled vs Experts...\n")

# Preparing demographics aligned with X_test
test_indices = X_test.index
df_test_demo = df_final.loc[test_indices, ['female', 'urbanicity', 'inc_q', 'age', 'country_code']].copy()

# Creating readable labels
df_test_demo['Gender'] = df_test_demo['female'].map({0: 'Male', 1: 'Female'})
df_test_demo['Location'] = df_test_demo['urbanicity'].map({0: 'Rural', 1: 'Urban'})
df_test_demo['Age_Group'] = pd.cut(
    df_test_demo['age'],
    bins=[0, 24, 34, 44, 54, 100],
    labels=['18-24', '25-34', '35-44', '45-54', '55+'],
    right=False
)


In [ ]:
# Functions with index reset
def get_subgroup_accuracy(y_true, y_pred, subgroup_col, subgroup_name, demo_df):
    # Resetting all indices to avoid misalignment
    y_true = pd.Series(y_true).reset_index(drop=True)
    y_pred = pd.Series(y_pred).reset_index(drop=True)
    demo_reset = demo_df.reset_index(drop=True)

    results = []
    for val in demo_reset[subgroup_col].dropna().unique():
        mask = (demo_reset[subgroup_col] == val)
        if mask.sum() < 10:
            continue

        label = str(val)
        acc = accuracy_score(y_true[mask], y_pred[mask])

        results.append({
            'Dimension': subgroup_name,
            'Subgroup': label,
            'Samples': mask.sum(),
            'Accuracy': acc
        })
    return pd.DataFrame(results)


In [ ]:
# Getting the Predictions
pooled_preds = model_pooled.predict(X_test)

expert_preds_dict = {}
for country, model in experts.items():
    mask = (country_test == country)
    if mask.sum() > 0:
        expert_preds_dict[country] = model.predict(X_test[mask])

# Calculating Accuracies
subgroup_results = []

dimensions = [
    ('Gender', 'Gender'),
    ('Location', 'Location'),
    ('Income Quintile', 'inc_q'),
    ('Age Group', 'Age_Group')
]

In [ ]:
# Pooled Model
for dim_name, col in dimensions:
    df_sg = get_subgroup_accuracy(y_test, pooled_preds, col, dim_name, df_test_demo)
    if not df_sg.empty:
        df_sg['Model'] = 'Pooled'
        subgroup_results.append(df_sg)

# Expert Models
for country, exp_preds in expert_preds_dict.items():
    mask_country = (country_test == country)
    y_c = y_test[mask_country]
    demo_c = df_test_demo.loc[mask_country]

    for dim_name, col in dimensions:
        df_sg = get_subgroup_accuracy(y_c, exp_preds, col, dim_name, demo_c)
        if not df_sg.empty:
            df_sg['Model'] = f'Expert_{country}'
            subgroup_results.append(df_sg)
# Combine and display
subgroup_df = pd.concat(subgroup_results, ignore_index=True).round(4)

print("Subgroup Accuracy Comparison (Pooled vs Experts):")
print(subgroup_df.to_string(index=False))

# Styled display
display(
    subgroup_df.style
    .format(precision=4)
    .highlight_max(color='lightgreen', subset=['Accuracy'])
    .set_caption("Accuracy by Subgroup: Pooled Model vs Country Experts"))

In [ ]:
# Visualization
plt.figure(figsize=(12, 6))
key_df = subgroup_df[subgroup_df['Dimension'].isin(['Gender', 'Location'])]

sns.barplot(
    data=key_df,
    x='Subgroup',
    y='Accuracy',
    hue='Model',
    palette='Set2'
)

plt.title('Accuracy by Gender and Location - Pooled vs Expert Models', fontsize=14)
plt.ylabel('Accuracy')
plt.ylim(0.85, 1.02)
plt.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

**Train Gating Model**

In [ ]:
# Trainning Gating Model
# Trainning a Random Forest to predict country_code

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

print("Training Gating Model (Country Router)...\n")

# Preparing  data for gating model
gating_X = X_train.copy()
gating_y = country_train.copy()          # Target = country_code (KEN, TZA, UGA)

print(f"Training gating model on {len(gating_X)} samples")
print(f"Countries to predict: {gating_y.unique()}")

In [ ]:
# Trainning the Gating Model
gating_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)

gating_model.fit(gating_X, gating_y)

print("Gating model trained successfully.")


In [ ]:
# Evaluating the Gating Model
print("\nGating Model Performance:")

# On training set
train_pred = gating_model.predict(gating_X)
train_acc = accuracy_score(gating_y, train_pred)

print(f"Training Accuracy: {train_acc:.4f}")


In [ ]:
# On test set (more realistic)
test_pred = gating_model.predict(X_test)
test_acc = accuracy_score(country_test, test_pred)

print(f"Test Accuracy    : {test_acc:.4f}")

# Detailed report
print("\nClassification Report (Test Set):")
print(classification_report(country_test, test_pred))


In [ ]:
# Feature Importance for Gating Model
print("\nTop 10 Features used by Gating Model:")
gating_importance = pd.Series(
    gating_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False).head(10)

print(gating_importance.round(4))

In [ ]:
# Save the gating model
import joblib
joblib.dump(gating_model, 'gating_model_rf.joblib')
print("\nGating model saved as 'gating_model_rf.joblib'")

**Simple Routing Function**

In [ ]:
# Routing Function using Gating Model
def predict_with_gating(X_input, return_routing_info=True):
    """
    Predict using gating model to route to correct expert,
    fallback to pooled if routing fails.
    """
    if isinstance(X_input, pd.DataFrame):
        X_input = X_input[X_train.columns]   # align columns

    # Get country prediction from gating model
    predicted_country = gating_model.predict(X_input)

    results = []
    routing_info = []

    for i, country in enumerate(predicted_country):
        sample = X_input.iloc[[i]] if isinstance(X_input, pd.DataFrame) else X_input[[i]]

        if country in experts:
            # Route to expert
            prob = experts[country].predict_proba(sample)[0, 1]
            model_used = f"Expert_{country}"
        else:
            # Fallback to pooled
            prob = model_pooled.predict_proba(sample)[0, 1]
            model_used = "Pooled"

        results.append(prob)
        routing_info.append(model_used)

    if return_routing_info:
        return np.array(results), routing_info
    else:
        return np.array(results)


print("\nGating model + routing function ready.")

**Dynamic Results Summary**

In [ ]:
# Dynamic Results Summary
# Automatically pulls metrics from df_results

import pandas as pd

print("Generating Dynamic Results Summary...\n")

# Ensure df_results exists
if 'df_results' not in globals():
    print("Error: df_results not found. Please run the evaluation cell first.")
else:
    # Extract key metrics
    pooled_all = df_results[df_results['Model'] == 'Pooled']
    pooled_all_f1 = pooled_all[pooled_all['Country'] == 'All']['F1'].values[0]
    pooled_all_auc = pooled_all[pooled_all['Country'] == 'All']['AUC'].values[0]

    # Per country metrics
    country_summary = []
    for country in ['KEN', 'TZA', 'UGA']:
        p_row = df_results[(df_results['Model'] == 'Pooled') & (df_results['Country'] == country)]
        e_row = df_results[(df_results['Model'].str.contains('Expert')) & (df_results['Country'] == country)]

        if not p_row.empty and not e_row.empty:
            p_f1 = p_row['F1'].values[0]
            p_auc = p_row['AUC'].values[0]
            e_f1 = e_row['F1'].values[0]
            e_auc = e_row['AUC'].values[0]

            winner = "Expert" if e_f1 > p_f1 + 0.005 else "Pooled" if p_f1 > e_f1 + 0.005 else "Tie"
            f1_diff = e_f1 - p_f1

            country_summary.append({
                'Country': country,
                'Pooled_F1': p_f1,
                'Expert_F1': e_f1,
                'Pooled_AUC': p_auc,
                'Expert_AUC': e_auc,
                'Winner': winner,
                'F1_Diff': f1_diff
            })

    # Generate dynamic markdown text
    markdown_text = f"""## Results Summary & Discussion

### Model Performance Comparison

The pooled model and country-specific expert models were evaluated on the held-out test set using **F1-score** and **AUC**.

**Overall Performance:**
- Pooled model (all countries): **F1 = {pooled_all_f1:.4f}**, AUC = {pooled_all_auc:.4f}

**Per-country breakdown:**

"""

    for cs in country_summary:
        diff_text = f"(+{cs['F1_Diff']:.4f})" if cs['F1_Diff'] > 0 else f"({cs['F1_Diff']:.4f})"
        markdown_text += f"- **{cs['Country']}**: Expert F1 = {cs['Expert_F1']:.4f} {diff_text} vs Pooled F1 = {cs['Pooled_F1']:.4f} → **{cs['Winner']} wins**\\n"

    markdown_text += f"""
**Verdict:**
The **expert models win** in countries with smaller data volumes (especially Tanzania and Uganda), while the **pooled model performs strongly in Kenya**.
This pattern suggests that country-specific models are particularly beneficial when per-country sample sizes are limited.

### Implications for Data Volume

This study reveals a clear **data volume effect**:

- In high-data countries (e.g. Kenya), the pooled model is efficient and competitive.
- In lower-data countries (Tanzania and Uganda), expert models consistently deliver better F1-scores by capturing local behavioral patterns that get diluted in the pooled approach.

**Key Insight:**
When data per country falls below a certain threshold, training separate expert models becomes advantageous for predictive performance in digital finance access modeling.

**Recommendation:**
For multi-country financial inclusion projects in East Africa, a **mixture-of-experts approach** (using a gating model to route to country experts, with pooled model as fallback) offers the best balance between accuracy and practicality.
"""

    # Display the generated markdown
    from IPython.display import Markdown, display
    display(Markdown(markdown_text))

    # Also print plain text version for easy copying
    print("\n" + "="*80)
    print("Plain text version (for copying):")
    print("="*80)
    print(markdown_text.replace("\\n", "\n"))

**RESULT SUMMARY,MANUAL ANALYSIS**

**Model Performance Comparison**

The pooled model and country-specific expert models were evaluated on the held-out test set using **F1-score** and **AUC** as primary metrics.

**Key Findings:**

- **Overall (across all countries):** The **pooled model** achieved competitive or slightly superior performance in most cases, benefiting from larger combined training data.
- **Per-country performance:**
  - In **Kenya** (largest sample): The pooled model performed marginally better or on par with the expert model.
  - In **Tanzania** and **Uganda** (smaller samples): The **country-specific expert models** consistently showed higher F1-scores and better AUC in several subgroups.

**Verdict:**  
The **expert models win in countries with smaller data volumes** (Tanzania and Uganda), while the **pooled model remains strong in Kenya** and when aggregating across countries. This demonstrates that when sufficient data is available, a single pooled model is efficient and robust. However, when data per country is limited, training separate expert models helps capture country-specific patterns that the pooled model tends to overlook.

**Implications for Data Volume**

This analysis highlights a clear relationship between **data volume** and modeling strategy:

- Countries with larger sample sizes (e.g., Kenya) allow the pooled model to generalize well, reducing the need for country specific models.
- In countries with relatively smaller samples (Tanzania and Uganda), expert models outperform the pooled approach by learning local nuances in digital finance adoption behavior.
- The performance gap suggests that **below a certain threshold of observations per country**, splitting into expert models becomes advantageous.

**Practical Recommendation:**  
For multi-country studies in financial inclusion, a **hybrid approach** is ideal, use a gating model to dynamically route predictions to the most suitable expert model, with the pooled model as a reliable fallback. This balances predictive performance with scalability, especially when expanding to more countries with varying data availability.

In conclusion, while a one-size-fits-all pooled model offers simplicity and stability, **country-specific expert models provide meaningful gains in predictive accuracy** when data volume per country is limited, a critical insight for building effective digital finance models across East Africa.

**Web Based Artefact**

In [ ]:
# installing Streamlit
!pip install streamlit shap -q

#Installing other dependencies if needed
!pip install matplotlib seaborn -q

In [ ]:
# app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt

st.set_page_config(page_title="Digital Finance Predictor", page_icon="💜", layout="wide")

# CUSTOM PURPLE THEME
st.markdown("""
    <style>
    .stApp {
        background: linear-gradient(135deg, #4C1D95 0%, #6B46C1 50%, #9F7AEA 100%);
        color: white;
    }
    .main .block-container {
        background: rgba(255, 255, 255, 0.13);
        border-radius: 20px;
        padding: 2.5rem;
        box-shadow: 0 10px 40px rgba(0, 0, 0, 0.4);
        backdrop-filter: blur(12px);
    }
    h1, h2, h3 { color: #E0BBFF; text-shadow: 0 2px 15px rgba(0,0,0,0.3); }
    .stButton>button {
        background: linear-gradient(90deg, #C4B5FD, #A78BFA);
        color: #2D1B69;
        font-weight: bold;
        border-radius: 12px;
        height: 3rem;
    }
    .result-box {
        background: rgba(255,255,255,0.15);
        border-radius: 15px;
        padding: 1.8rem;
        border: 1px solid rgba(255,255,255,0.25);
    }
    </style>
""", unsafe_allow_html=True)

st.title("💜 Digital Finance Access Predictor")
st.markdown("### East Africa • Kenya | Tanzania | Uganda")

st.markdown("---")


In [ ]:
# Loading models
@st.cache_resource
def load_models():
    try:
        pooled = joblib.load('trained_models/model_pooled_rf.joblib')
        experts = {
            'KEN': joblib.load('trained_models/expert_model_KEN.joblib'),
            'TZA': joblib.load('trained_models/expert_model_TZA.joblib'),
            'UGA': joblib.load('trained_models/expert_model_UGA.joblib')
        }
        gating = joblib.load('trained_models/gating_model_rf.joblib')
        return pooled, experts, gating
    except Exception as e:
        st.error(f"Error loading models: {e}")
        return None, None, None

model_pooled, experts, gating_model = load_models()


In [ ]:
# INPUT SECTION
st.subheader("👤 Enter User Profile")

col1, col2, col3 = st.columns(3)

with col1:
    age = st.slider("Age", 18, 80, 32)
    female = st.radio("Gender", ["Male", "Female"], horizontal=True)
    urbanicity = st.radio("Location", ["Rural", "Urban"], horizontal=True)

with col2:
    inc_q = st.selectbox("Income Quintile", [1,2,3,4,5])
    educ = st.selectbox("Education Level",
                        options=[0,1,2,3,4],
                        format_func=lambda x: ["No Education","Primary","Secondary","Tertiary","Higher"][x])
    internet_use = st.radio("Uses Internet", ["No", "Yes"], horizontal=True)

with col3:
    country = st.selectbox("Country", ["KEN (Kenya)", "TZA (Tanzania)", "UGA (Uganda)"])
    country_code = country.split()[0]

In [ ]:
# Preparing input data
input_dict = {
    'age': age,
    'female': 1 if female == "Female" else 0,
    'urbanicity': 1 if urbanicity == "Urban" else 0,
    'inc_q': inc_q,
    'educ': educ,
    'internet_use': 1 if internet_use == "Yes" else 0,
}

input_df = pd.DataFrame([input_dict])

# Align with training columns
for col in X_train.columns:
    if col not in input_df.columns:
        input_df[col] = 0
input_df = input_df[X_train.columns]


In [ ]:
import joblib

# Save the trained models as .pkl files
joblib.dump(gating_model, 'gating_model.pkl')
joblib.dump(experts, 'experts.pkl')

print("Created successfully:")
!ls -l gating_model.pkl experts.pkl

In [ ]:
#  Clone repo
!git clone https://github.com/sitahlango-maker/Financial_Inclusion.git
%cd /content/Financial_Inclusion/Colab\ Notebooks/FinancialInclution

In [ ]:
# Making sure we're in the cloned repo folder
%cd /content/Financial_Inclusion/Colab\ Notebooks/FinancialInclution

In [ ]:
# Setting Git identity
!git config --global user.email "sitah.lango@gmail.com"
!git config --global user.name "sitahlango-maker"

In [ ]:
import os

def search_for_file(filename, start_dirs=None):
    """
    Search for a file starting from one or more directories.
    Returns list of full paths where the file is found.
    """
    if start_dirs is None:
        start_dirs = ['/content', os.getcwd(), os.path.expanduser('~')]

    found = []
    for start in start_dirs:
        if not os.path.isdir(start):
            continue
        for root, dirs, files in os.walk(start):
            if filename in files:
                full_path = os.path.join(root, filename)
                found.append(full_path)
                print(f"Found: {full_path}")
    return found

# Running the search
print("=== Searching for gating_model.pkl ===")
paths_g = search_for_file("gating_model.pkl")

print("\n=== Searching for experts.pkl ===")
paths_e = search_for_file("experts.pkl")

if not paths_g and not paths_e:
    print("\n→ No matching .pkl files were found in the searched locations.")
    print("Common places to check manually:")
    print("  - Google Drive:      /content/drive/MyDrive/  or /content/drive/MyDrive/Thesis/")
    print("  - Local folder:      where you trained/saved the models")
    print("  - Downloads:         ~/Downloads/")
    print("\nTip: If models are in Google Drive, mount it first:")
    print("from google.colab import drive")
    print("drive.mount('/content/drive')")
    print("Then re-run the search (it will include /content/drive)")

In [ ]:
import requests

# Raw GitHub URLs (these are the direct download links)
urls = {
    "gating_model.pkl": "https://raw.githubusercontent.com/sitahlango-maker/Financial_Inclusion/main/Colab%20Notebooks/FinancialInclution/gating_model.pkl",
    "experts.pkl":      "https://raw.githubusercontent.com/sitahlango-maker/Financial_Inclusion/main/Colab%20Notebooks/FinancialInclution/experts.pkl"
}

for name, url in urls.items():
    print(f"\nChecking: {name}")
    try:
        response = requests.head(url, allow_redirects=True, timeout=8)
        status = response.status_code

        if status == 200:
            size = int(response.headers.get('Content-Length', 0))
            content_type = response.headers.get('Content-Type', 'unknown')

            if size > 0 and 'application/octet-stream' in content_type or size > 1000:
                print("→ File EXISTS and appears downloadable")
                print(f"   Size: {size:,} bytes")
                print(f"   Content-Type: {content_type}")
            else:
                print("→ Exists but suspicious (very small or wrong type)")
        elif status == 404:
            print("→ File NOT FOUND (404)")
        else:
            print(f"→ Unexpected status: {status}")
            print(f"   Response: {response.text[:200]}...")

    except requests.exceptions.RequestException as e:
        print(f"→ Error checking URL: {e}")

In [ ]:
# Confirming current location & files
!pwd
!ls -la

In [ ]:
# Adding model files
!git add gating_model.pkl experts.pkl

In [ ]:
print(df_final.columns.tolist())

In [ ]:
joblib.dump(X_train.median(numeric_only=True).to_dict(), 'medians.pkl')

In [ ]:
joblib.dump(list(X_train.columns), 'feature_names.pkl')

In [ ]:
# app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import os

st.set_page_config(page_title="Digital Finance Predictor", layout="wide")

st.title("💜 Digital Finance Access Predictor")

# -------------------------------
# LOAD MODELS (robust path logic)
# -------------------------------
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except:
    BASE_DIR = os.getcwd()

MODEL_DIR = os.path.join(BASE_DIR, "trained_models")

try:
    model_pooled = joblib.load(os.path.join(MODEL_DIR, "model_pooled.pkl"))
    gating_model = joblib.load(os.path.join(MODEL_DIR, "gating_model.pkl"))
    experts = joblib.load(os.path.join(MODEL_DIR, "experts.pkl"))
    feature_names = joblib.load(os.path.join(MODEL_DIR, "feature_names.pkl"))
    st.success("✅ Models loaded successfully")
except Exception as e:
    st.error(f"❌ Error loading models: {e}")
    st.stop()

# -------------------------------
# USER INPUT UI
# -------------------------------
st.sidebar.header("Enter User Profile")

input_data = {}

for feature in feature_names:
    input_data[feature] = st.sidebar.number_input(feature, value=0.0)

input_df = pd.DataFrame([input_data])

# -------------------------------
# PREDICTION
# -------------------------------
if st.button("🔮 Predict", use_container_width=True):

    with st.spinner("Running Mixture of Experts..."):

        try:
            pred_country = gating_model.predict(input_df)[0]
            gating_conf = gating_model.predict_proba(input_df).max(axis=1)[0]

            if pred_country in experts and gating_conf >= 0.40:
                final_model = experts[pred_country]
                model_used = f"Expert ({pred_country})"
            else:
                final_model = model_pooled
                model_used = "Pooled Model"

            prob = final_model.predict_proba(input_df)[0, 1]

            # -------------------------------
            # DISPLAY RESULTS
            # -------------------------------
            st.subheader("🎯 Prediction Result")

            col1, col2 = st.columns(2)

            with col1:
                st.metric("Probability of Access", f"{prob:.2%}")

            with col2:
                st.metric("Model Used", model_used)
                st.metric("Gating Confidence", f"{gating_conf:.2%}")

            # Status message
            if prob >= 0.75:
                st.success("🟢 High Chance")
            elif prob >= 0.5:
                st.info("🔵 Moderate Chance")
            else:
                st.warning("🟠 Low Chance")

        except Exception as e:
            st.error(f"Prediction error: {e}")

**Running it Publicly**

In [ ]:
!pip install -q streamlit pyngrok

In [ ]:
!ngrok config add-authtoken 3B4zf0P1DMYB3xtG7NqvCtvT3TG_7jaZE9nanv58HumJdpejp


In [ ]:
import subprocess
import time
from pyngrok import ngrok

# 1. Kill any previous Streamlit or ngrok processes (prevents port conflicts)
!pkill -f streamlit 2>/dev/null || true
!pkill -f ngrok 2>/dev/null || true

print("🚀 Starting Streamlit server in the background...")

# 2. Start Streamlit using subprocess (this is the key part)
subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.headless", "true",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false"
])

# 3. Give Streamlit enough time to fully start and bind to port 8501
print("⏳ Waiting for Streamlit to initialize (8–12 seconds)...")
time.sleep(10)   # ← Increase to 12 or 15 if your app loads heavy models

# 4. Optional: Check if Streamlit is really running
!lsof -i :8501 || echo "⚠️  Warning: No process found on port 8501"

# 5. Create the public ngrok URL
public_url = ngrok.connect(8501, proto="http", bind_tls=True)
print("✅ Your Streamlit app is now live!")
print(public_url)

**Launch**

In [ ]:
#Create a f public URL
!pkill -f streamlit 2>/dev/null
!pkill -f ngrok    2>/dev/null

!streamlit run app.py &>/content/logs.txt &

from pyngrok import ngrok
public_url = ngrok.connect(8501)
print("New App URL:", public_url)


In [ ]:
!ps aux | grep streamlit

In [ ]:
!cat /content/logs.txt

In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv('final_combined_data.csv')

# Basic definition checks
print("Shape:", df.shape)          # (rows, columns)
print("\nColumns:\n", df.columns)  # column names
print("\nPreview:\n", df.head())   # first rows

In [ ]:
# Dataset definition
dataset_name = "final_combined_data.csv"

print(f"""
Dataset: {dataset_name}

Description:
This dataset is a cleaned and merged analytical dataset combining
individual-level financial inclusion data (Global Findex 2021)
with relevant macro and contextual indicators.

Unit of Analysis:
Individual respondents (age 15+)

Time Frame:
Cross-sectional data for the year 2021

Purpose:
To analyse determinants and patterns of digital financial services adoption
in East Africa.

Dimensions:
- Rows: {df.shape[0]} individuals
- Columns: {df.shape[1]} variables
""")

In [ ]:
# Check available columns first (just to confirm)
print(df.columns)

# Define target column (adjust if needed)
target_col = 'dig_account'

# Distribution (counts)
dig_account = df[target_col].value_counts()

# Distribution (percentages)
percentages = df[target_col].value_counts(normalize=True) * 100

# Combine into one table
distribution = dig_account.to_frame(name='Count')
distribution['Percentage (%)'] = percentages

print(distribution)

In [ ]:
import matplotlib.pyplot as plt

# Map labels clearly
label_map = {1: 'Has Access', 0: 'No Access'}

# Convert index values to readable labels
labels = [label_map.get(i, str(i)) for i in distribution.index]

plt.figure()

plt.pie(
    dig_account,
    labels=labels,
    autopct='%1.1f%%',
    startangle=90
)

plt.title('Distribution of Financial Inclusion (Account Ownership)')
plt.axis('equal')
plt.show()

In [ ]:
!pip install geopandas matplotlib

In [ ]:
country_mean = df.groupby('country_code')['dig_account'].mean().reset_index()

# Convert to percentage
country_mean['inclusion_rate'] = country_mean['dig_account'] * 100

print(country_mean)

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

# Your data (already computed)
country_mean = df.groupby('country_code')['dig_account'].mean().reset_index()
country_mean['inclusion_rate'] = country_mean['dig_account'] * 100

# Map ISO codes to Natural Earth country names
code_to_name = {
    'KEN': 'Kenya',
    'TZA': 'Tanzania',
    'UGA': 'Uganda'
}

country_mean['country'] = country_mean['country_code'].map(code_to_name)

print(country_mean)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

# Load Natural Earth dataset from official source
world = gpd.read_file(
    "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
)

In [ ]:
africa = world[world['CONTINENT'] == 'Africa']

In [ ]:
country_mean = df.groupby('country_code')['dig_account'].mean().reset_index()
country_mean['inclusion_rate'] = country_mean['dig_account'] * 100

code_to_name = {
    'KEN': 'Kenya',
    'UGA': 'Uganda',
    'TZA': 'United Republic of Tanzania'
}

country_mean['country'] = country_mean['country_code'].map(code_to_name)

In [ ]:
merged = africa.merge(
    country_mean,
    left_on='ADMIN',
    right_on='country',
    how='left'
)

In [ ]:
plt.figure(figsize=(8,6))

merged.plot(
    column='inclusion_rate',
    cmap='OrRd',
    legend=True,
    edgecolor='black',
    missing_kwds={"color": "lightgrey"}
)

plt.title('Financial Inclusion Heatmap (East Africa)')
plt.axis('off')

plt.show()

In [ ]:
merged['coords'] = merged['geometry'].representative_point()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(10, 6))

# --- Choropleth map ---
merged.plot(
    column='inclusion_rate',
    cmap='OrRd',
    edgecolor='black',
    missing_kwds={"color": "lightgrey"},
    ax=ax,
    legend=False
)

# --- Colorbar (main % scale) ---
sm = plt.cm.ScalarMappable(
    cmap='OrRd',
    norm=plt.Normalize(
        vmin=merged['inclusion_rate'].min(),
        vmax=merged['inclusion_rate'].max()
    )
)

cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.04)
cbar.set_label('Financial Inclusion (%)')

# --- Country labels on map ---
merged['coords'] = merged['geometry'].representative_point()

for idx, row in merged.dropna(subset=['inclusion_rate']).iterrows():
    ax.text(
        row['coords'].x,
        row['coords'].y,
        f"{row['inclusion_rate']:.1f}%",
        ha='center',
        fontsize=9,
        fontweight='bold'
    )

# --- Custom legend (exact country means) ---
legend_labels = [
    mpatches.Patch(color='none', label='Country Means'),
    mpatches.Patch(color='none', label='--------------------'),
    mpatches.Patch(color='none', label='Kenya: 90.8%'),
    mpatches.Patch(color='none', label='Uganda: 76.3%'),
    mpatches.Patch(color='none', label='Tanzania: 61.1%')
]

ax.legend(
    handles=legend_labels,
    loc='center left',
    bbox_to_anchor=(1.15, 0.5),
    frameon=True
)

ax.set_title('Financial Inclusion in East Africa (%)')
ax.axis('off')

plt.tight_layout()
plt.show()